# Estoque de carbono do solo e paisagem virtual (n mínimo de amostras)

Notebook em etapas. Rode as células em ordem.

**Parte A – dados observados**
- A1. Análise descritiva por área e camada (média, DP, CV, normalidade)
- A2. Boxplots por camada com teste de Kruskal-Wallis e Dunn (Bonferroni)
- A3. Mapas de interpolação (spline de placas finas) por área e camada

**Parte B – paisagem virtual**
- B1. Leitura dos ortomosaicos e cálculo dos índices em cada pixel
- B2. Sinal espectral, R² observado e variograma, para cada camada
- B3. Simulação do ECOS em cada pixel (campo aleatório gaussiano)
- B3b. Figura da paisagem real × paisagem virtual
- B4. Amostragem virtual e treino dos modelos
- B5. n mínimo e tabela
- B6. Curvas de aprendizado (em inglês)

**Parte C – relevo (MDE de 30 m)**
- C1. MDE do Copernicus (baixa uma vez e guarda na pasta `relevo`)
- C2. Atributos do relevo nos pontos (declividade, curvatura, TPI, TWI)
- C3. Correlação e validação cruzada: o relevo explica o ECOS?

Siglas nas figuras (inglês): NV = native vegetation (VN), ICL = integrated crop-livestock (ILP), ILF = integrated livestock-forest (IPF).

Na primeira vez, instale as bibliotecas (célula abaixo). Depois pode deixar comentada.

In [ ]:
# %pip install rasterio geopandas gstools scikit-learn matplotlib scipy

## Configuração
Os caminhos são relativos à pasta `notebooks/` do repositório. Para um teste rápido, use `REPS = 2`; para o resultado final, `REPS = 30`.

In [ ]:
import os, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.enums import Resampling
from shapely import contains_xy
from scipy import stats
from scipy.spatial import cKDTree
from scipy.stats import norm
from scipy.interpolate import RBFInterpolator
import gstools as gs
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
warnings.filterwarnings('ignore')

DADOS = os.path.join('..', 'data')            # pasta de dados do repositório
OBS = os.path.join(DADOS, 'soc_points.csv')
SAIDA = os.path.join('..', 'results', 'notebook_run')
os.makedirs(SAIDA, exist_ok=True)

# raster multiespectral de cada área (~1,25 m; bandas: azul, verde, pan, vermelho, red edge, NIR) e pontos amostrais
AREAS = {a: (os.path.join(DADOS, 'rasters', f'{a}_1m.tif'), os.path.join(DADOS, 'points', f'{a}_points.gpkg'))
         for a in ('VN', 'ILP', 'IPF')}
SIGLA_EN = {'VN': 'NV', 'ILP': 'ICL', 'IPF': 'ILF'}
NOME_EN = {'VN': 'Native vegetation (NV)', 'ILP': 'Integrated crop-livestock (ICL)',
           'IPF': 'Integrated livestock-forest (ILF)'}
CAMADAS = ['solo0-10', 'solo10-20', 'solo20-30', 'solo0-30']
ROT = {'solo0-10': '0–10 cm', 'solo10-20': '10–20 cm', 'solo20-30': '20–30 cm', 'solo0-30': '0–30 cm'}
ALVOS = CAMADAS              # camadas simuladas na Parte B (cada uma ~1/4 do tempo total)
PIX = 2.5                    # tamanho do pixel da paisagem virtual (m)
MARGEM = 15                  # margem em torno da grade amostral (m)
NS = [10, 20, 30, 50, 75, 100, 150, 200, 300]
CENARIOS = ['observado', 0.3, 0.5, 0.7]   # força do sinal (R²)
REPS = 30                    # repetições por n e cenário
N_AVAL = 5000                # pixels usados para avaliar contra o mapa verdadeiro
FRAC, PROB = 0.8, 0.8        # n mínimo: P(R² >= FRAC * R²_teto) >= PROB
SEED = 2024
IDX = ['EVI', 'SAVI', 'NDVI', 'NDRE', 'PRI', 'sPRI', 'CO2FLUX', 'NDWI', 'SOCI', 'SI', 'CI', 'OI']
rng = np.random.default_rng(SEED)
obs_all = pd.read_csv(OBS)

# estilo das figuras
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.linewidth': .6,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 400,
                     'axes.titlesize': 7.5, 'legend.fontsize': 6.5})
COR = {'VN': '#2a78d6', 'ILP': '#eb6834', 'IPF': '#1baf7a'}
SOC = LinearSegmentedColormap.from_list('soc', ['#f6efe3', '#d9b98a', '#a8743f', '#6b4423', '#3b2412'])
INK, MUTED = '#222222', '#6b6b6b'

## Funções

In [ ]:
def indices(B, G, R, RE, N):
    # índices espectrais conforme a Tabela 1 do manuscrito
    ndvi = (N - R) / (N + R)
    pri = (G - B) / (G + B)
    spri = (pri + 1) / 2
    return {'EVI': 2.5 * (N - R) / (N + 6 * R - 7.5 * B + 1),
            'SAVI': 1.5 * (N - R) / (N + R + 0.5),
            'NDVI': ndvi, 'NDRE': (N - RE) / (N + RE), 'PRI': pri, 'sPRI': spri,
            'CO2FLUX': ndvi * spri, 'NDWI': (G - N) / (G + N),
            'SOCI': B / (R * G), 'SI': (R - B) / (R + B),
            'CI': (R - G) / (R + G), 'OI': (2 * R - G - B) / (R + G)}


def anamorfose(z, alvo):
    # transforma valores normais padrão para a distribuição empírica dos observados
    q = np.sort(alvo)
    p = (np.arange(1, len(q) + 1) - 0.5) / len(q)
    return np.interp(norm.cdf(z), p, q)


def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 30)))


def dunn_bonferroni(grupos):
    # teste de Dunn (com correção de empates) e ajuste de Bonferroni; grupos: dict nome -> valores
    nomes = list(grupos)
    x = np.concatenate([grupos[k] for k in nomes])
    lab = np.repeat(nomes, [len(grupos[k]) for k in nomes])
    r = stats.rankdata(x); N = len(x)
    _, t = np.unique(x, return_counts=True)
    emp = (t ** 3 - t).sum() / (12 * (N - 1))
    m = len(nomes) * (len(nomes) - 1) / 2
    p = {}
    for i in range(len(nomes)):
        for j in range(i + 1, len(nomes)):
            a, b = nomes[i], nomes[j]
            na, nb = (lab == a).sum(), (lab == b).sum()
            z = (r[lab == a].mean() - r[lab == b].mean()) / np.sqrt((N * (N + 1) / 12 - emp) * (1 / na + 1 / nb))
            p[(a, b)] = p[(b, a)] = min(2 * norm.sf(abs(z)) * m, 1.0)
    return p


def letras(medias, p, alfa=0.05):
    # letras de agrupamento (a = maior média); grupos com a mesma letra não diferem
    ordem = sorted(medias, key=medias.get, reverse=True)
    conj = []
    for g in ordem:
        entrou = False
        for S in conj:
            if all(p[(g, h)] >= alfa for h in S):
                S.add(g); entrou = True
        if not entrou:
            S = {g}
            for h in ordem[:ordem.index(g)]:
                if all(p[(h, k)] >= alfa for k in S):
                    S.add(h)
            conj.append(S)
    return {g: ''.join('abcdefgh'[i] for i, S in enumerate(conj) if g in S) for g in medias}


def classe_cv(cv):
    # Warrick e Nielsen (1980)
    return 'baixo' if cv < 12 else ('médio' if cv <= 60 else 'alto')

# Parte A – dados observados
## A1. Análise descritiva
Média, desvio padrão, mediana, mínimo, máximo, coeficiente de variação (CV, classificado segundo Warrick e Nielsen, 1980) e teste de Shapiro-Wilk, por área e camada. A última coluna traz o p do teste de Kruskal-Wallis entre áreas.

In [ ]:
linhas = []
for cam in CAMADAS:
    kw = stats.kruskal(*[obs_all.loc[obs_all.tratamento == a, cam] for a in AREAS]).pvalue
    for a in AREAS:
        v = obs_all.loc[obs_all.tratamento == a, cam].dropna()
        cv = 100 * v.std() / v.mean()
        linhas.append({'Camada': ROT[cam], 'Área': a, 'n': len(v), 'Média': v.mean(), 'DP': v.std(),
                       'Mediana': v.median(), 'Mín': v.min(), 'Máx': v.max(), 'CV (%)': cv,
                       'Classe CV': classe_cv(cv), 'Shapiro p': stats.shapiro(v).pvalue, 'Kruskal p': kw})
desc = pd.DataFrame(linhas)
desc.to_csv(os.path.join(SAIDA, 'A1_descritiva.csv'), index=False, encoding='utf-8-sig')
desc.round(3)

## A2. Boxplots por camada
Um painel por camada (0-10, 10-20, 20-30 e 0-30 cm), com as três áreas lado a lado. Caixa = intervalo interquartil, linha = mediana, losango = média, pontos = observações. Letras: teste de Dunn com ajuste de Bonferroni (p < 0,05). Abaixo de cada caixa, o CV.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(7.2, 2.9), constrained_layout=True)
letras_tab = []
for k, (A, cam) in enumerate(zip(axs, CAMADAS)):
    grupos = {a: obs_all.loc[obs_all.tratamento == a, cam].dropna().values for a in AREAS}
    p = dunn_bonferroni(grupos)
    let = letras({a: np.mean(v) for a, v in grupos.items()}, p)
    topo = max(v.max() for v in grupos.values()); base = min(v.min() for v in grupos.values())
    for i, (a, v) in enumerate(grupos.items()):
        A.boxplot(v, positions=[i], widths=.55, patch_artist=True, showfliers=False,
                  boxprops=dict(facecolor=COR[a] + '33', edgecolor=COR[a], lw=1),
                  medianprops=dict(color=COR[a], lw=1.4), whiskerprops=dict(color=COR[a], lw=1),
                  capprops=dict(color=COR[a], lw=1))
        A.scatter(i + rng.uniform(-.13, .13, len(v)), v, s=6, color=COR[a], alpha=.55, lw=0, zorder=3)
        A.scatter(i, v.mean(), marker='D', s=16, color='#ffffff', edgecolor=INK, lw=.7, zorder=4)
        A.text(i, v.max() + .04 * (topo - base), let[a], ha='center', va='bottom', fontsize=7.5, fontweight='bold')
        A.text(i, base - .12 * (topo - base), f'CV {100 * v.std(ddof=1) / v.mean():.1f}%', ha='center', fontsize=5.8, color=MUTED)
        letras_tab.append({'Camada': ROT[cam], 'Área': a, 'Letra': let[a]})
    A.set_ylim(base - .18 * (topo - base), topo + .15 * (topo - base))
    A.set_xticks(range(3)); A.set_xticklabels([SIGLA_EN[a] for a in AREAS])
    A.set_title(ROT[cam], loc='left'); A.text(-.02, 1.03, 'abcd'[k], transform=A.transAxes, fontsize=9, fontweight='bold', ha='right')
    if k == 0: A.set_ylabel('SOC stock (Mg ha⁻¹)')
    A.grid(axis='y', color='#e2e2df', lw=.5); A.set_axisbelow(True)
fig.text(.01, -.04, 'NV, native vegetation; ICL, integrated crop-livestock; ILF, integrated livestock-forest. '
         'Diamond, mean; different letters, Dunn test with Bonferroni adjustment (p < 0.05).', fontsize=5.8, color=MUTED)
fig.savefig(os.path.join(SAIDA, 'A2_boxplots_SOC.png'), bbox_inches='tight')
fig.savefig(os.path.join(SAIDA, 'A2_boxplots_SOC.pdf'), bbox_inches='tight')
plt.show()
pd.DataFrame(letras_tab).pivot(index='Área', columns='Camada', values='Letra')

## A3. Mapas de interpolação (um por área)
Spline de placas finas (thin plate spline, `scipy.interpolate.RBFInterpolator`, suavização = 1), como no manuscrito, numa grade de 2 m dentro da área amostrada (envoltória dos pontos + 15 m). Uma figura por área, com as quatro camadas e coordenadas UTM (SIRGAS 2000 / UTM 23S). Com 20 a 26 pontos por área, os mapas são descritivos.

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator
fmt = FuncFormatter(lambda v, _: f'{v:,.0f}'.replace(',', ' '))
PONTOS = {}
for a, (_, shp) in AREAS.items():
    g = gpd.read_file(shp).to_crs(31983)
    PONTOS[a] = g.assign(**obs_all[obs_all.tratamento == a].reset_index(drop=True)[CAMADAS])

for a in AREAS:
    g = PONTOS[a]; xy = np.c_[g.geometry.x, g.geometry.y]
    dom = g.union_all().convex_hull.buffer(MARGEM)
    x0, y0, x1, y1 = dom.bounds
    gx, gy = np.meshgrid(np.arange(x0, x1, 2.0), np.arange(y1, y0, -2.0))
    dentro = contains_xy(dom, gx.ravel(), gy.ravel())
    larg = (x1 - x0) / (y1 - y0)
    if larg > 2:   # área alongada: uma camada por linha
        nl, nc = 4, 1; alt = 4 * 6.2 / larg + 1.2
    else:
        nl, nc = 2, 2; alt = 2 * 3.1 / larg + 1.2
    fig, axs = plt.subplots(nl, nc, figsize=(7.2, alt), constrained_layout=True)
    for k, (A, cam) in enumerate(zip(axs.ravel(), CAMADAS)):
        tps = RBFInterpolator(xy, g[cam].values, kernel='thin_plate_spline', smoothing=1.0)
        z = np.full(gx.size, np.nan)
        z[dentro] = tps(np.c_[gx.ravel()[dentro], gy.ravel()[dentro]])
        im = A.imshow(z.reshape(gx.shape), extent=[x0, x1, y0, y1], cmap=SOC)
        A.scatter(xy[:, 0], xy[:, 1], s=6, c='#ffffff', edgecolors=INK, linewidths=.5)
        A.xaxis.set_major_formatter(fmt); A.yaxis.set_major_formatter(fmt)
        A.xaxis.set_major_locator(MaxNLocator(5)); A.yaxis.set_major_locator(MaxNLocator(3))
        A.tick_params(labelsize=5.5); plt.setp(A.get_yticklabels(), rotation=90, va='center')
        A.spines[['top', 'right']].set_visible(True)
        A.set_title(f'{"abcd"[k]}   {ROT[cam]}', loc='left', fontweight='bold')
        if k >= 4 - nc: A.set_xlabel('Easting (m)')
        if k % nc == 0: A.set_ylabel('Northing (m)')
        A.annotate('N', xy=(0.04, 0.95), xytext=(0.04, 0.72), xycoords='axes fraction', ha='center', fontsize=6.5,
                   fontweight='bold', arrowprops=dict(arrowstyle='-|>', color=INK, lw=.8))
        cb = fig.colorbar(im, ax=A, shrink=.8, pad=.02)
        cb.set_label('SOC stock (Mg ha⁻¹)', fontsize=6); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=5.5)
    fig.suptitle(f'{NOME_EN[a]} – SOC stocks interpolated by thin plate spline (SIRGAS 2000 / UTM zone 23S)', fontsize=7.5)
    nome = SIGLA_EN[a]
    fig.savefig(os.path.join(SAIDA, f'A3_map_SOC_{nome}.png'), bbox_inches='tight')
    fig.savefig(os.path.join(SAIDA, f'A3_map_SOC_{nome}.pdf'), bbox_inches='tight')
    plt.show()

# Parte B – paisagem virtual
## B1. Ortomosaicos e índices em cada pixel
Lê o raster multiespectral de cada área reamostrado para 2,5 m, recorta a região da grade amostral e calcula os índices. No fim, confere os índices nos pontos com a planilha de dados.

In [ ]:
PAIS = {}
for area, (raster, pontos) in AREAS.items():
    with rasterio.open(raster) as src:
        f = src.res[0] / PIX
        h, w = int(src.height * f), int(src.width * f)
        a = src.read(out_shape=(6, h, w), resampling=Resampling.average, masked=True)
        tr = src.transform * src.transform.scale(src.width / w, src.height / h)
        crs = src.crs
    g = gpd.read_file(pontos).to_crs(crs)
    dom = g.union_all().convex_hull.buffer(MARGEM)
    cols, rows = np.meshgrid(np.arange(w), np.arange(h))
    xs, ys = rasterio.transform.xy(tr, rows.ravel(), cols.ravel())
    xs, ys = np.asarray(xs), np.asarray(ys)
    B, G, _, R, RE, N = [a[i].filled(np.nan).ravel().astype(float) for i in range(6)]
    ok = contains_xy(dom, xs, ys) & np.isfinite(B) & (B > 0) & (N > 0)
    I = pd.DataFrame(indices(B[ok], G[ok], R[ok], RE[ok], N[ok]))
    I['x'], I['y'] = xs[ok], ys[ok]
    I['bR'], I['bG'], I['bB'] = R[ok], G[ok], B[ok]   # cor verdadeira, só para a figura B3b
    I = I.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    _, k = cKDTree(I[['x', 'y']].values).query(np.c_[g.geometry.x, g.geometry.y])
    obs = obs_all[obs_all.tratamento == area].reset_index(drop=True)
    PAIS[area] = dict(I=I, pontos=g, k=k, obs=obs)
    conf = {v: np.corrcoef(I.iloc[k][v], obs[v])[0, 1] for v in ['NDVI', 'SOCI', 'OI']}
    print(f'{area}: {len(I)} pixels ({len(I) * PIX**2 / 1e4:.1f} ha); '
          f'conferência com a planilha (r): ' + ', '.join(f'{v} {r:.2f}' for v, r in conf.items()))

## B2. Sinal espectral, R² observado e variograma (por camada)
- Direção do sinal: combinação linear dos índices estimada nos pontos observados (ridge), padronizada.
- R² observado: correlação ao quadrado entre a predição fora da amostra (leave-one-out) e o ECOS medido.
- Alcance e pepita do resíduo espacial: variograma exponencial dos resíduos nos pontos (limitado entre 10 e 300 m).

In [ ]:
param = []
for area, P in PAIS.items():
    I, obs, k, g = P['I'], P['obs'], P['k'], P['pontos']
    sc = StandardScaler().fit(I[IDX].values)
    Xp, XI = sc.transform(I.iloc[k][IDX].values), sc.transform(I[IDX].values)
    pos = np.c_[g.geometry.x, g.geometry.y]
    bins = np.linspace(0, np.ptp(pos, 0).max() * 0.6, 8)
    P['alvo'] = {}
    for alvo in ALVOS:
        yp = obs[alvo].values
        rid = RidgeCV(alphas=np.logspace(-2, 4, 30)).fit(Xp, yp)
        sinal = XI @ rid.coef_
        sinal = (sinal - sinal.mean()) / sinal.std() if sinal.std() > 0 else np.zeros(len(I))
        pr = cross_val_predict(RidgeCV(alphas=np.logspace(-2, 4, 30)), Xp, yp, cv=LeaveOneOut())
        r2o = max(np.corrcoef(pr, yp)[0, 1], 0) ** 2
        try:
            bc, gam = gs.vario_estimate((pos[:, 0], pos[:, 1]), yp - rid.predict(Xp), bins)
            mod = gs.Exponential(dim=2); mod.fit_variogram(bc, gam, nugget=True)
            alc, pep = float(np.clip(mod.len_scale, 10, 300)), float(mod.nugget / (mod.nugget + mod.var))
        except Exception:
            alc, pep = 50.0, 0.3
        P['alvo'][alvo] = dict(sinal=sinal, r2_obs=r2o, alcance=alc, pepita=pep)
        param.append(dict(area=area, camada=ROT[alvo], pixels=len(I), area_ha=round(len(I) * PIX**2 / 1e4, 1),
                          R2_observado=round(r2o, 3), alcance_m=round(alc, 1), pepita_rel=round(pep, 2)))
param = pd.DataFrame(param); param.to_csv(os.path.join(SAIDA, 'B2_parametros_paisagem.csv'), index=False)
param

## B3. Simulação do ECOS em cada pixel
Para cada camada e cenário: `z = sqrt(R²) * sinal + sqrt(1 - R²) * resíduo`, em que o resíduo é um campo aleatório gaussiano (covariância exponencial + pepita). Depois, `z` é transformado para a distribuição dos estoques observados da área naquela camada. `teto` é o R² máximo possível no cenário.

In [ ]:
for area, P in PAIS.items():
    I = P['I']
    for alvo, Q in P['alvo'].items():
        Q['y'], Q['teto'] = {}, {}
        for cen in CENARIOS:
            r2 = Q['r2_obs'] if cen == 'observado' else cen
            srf = gs.SRF(gs.Exponential(dim=2, var=1, len_scale=Q['alcance']), seed=int(rng.integers(1e9)))
            campo = srf((I.x.values, I.y.values), mesh_type='unstructured')
            campo = (campo - campo.mean()) / campo.std()
            e = np.sqrt(1 - Q['pepita']) * campo + np.sqrt(Q['pepita']) * rng.standard_normal(len(I))
            e = (e - e.mean()) / e.std()
            z = np.sqrt(r2) * Q['sinal'] + np.sqrt(1 - r2) * e
            y = anamorfose((z - z.mean()) / z.std(), P['obs'][alvo].values)
            Q['y'][str(cen)] = y
            Q['teto'][str(cen)] = np.corrcoef(Q['sinal'], y)[0, 1] ** 2 if r2 > 0 else 0.0
    print(area, 'ok')

## B3b. Figura: paisagem real × paisagem virtual (0-30 cm)
À esquerda, a imagem do ARP em cor verdadeira com o ECOS medido nos pontos; à direita, o ECOS simulado em cada pixel no cenário observado. As duas colunas usam a mesma escala de cor.

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator
import scipy.ndimage as ndi
fmt = FuncFormatter(lambda v, _: f'{v:,.0f}'.replace(',', ' '))
ALVO_FIG = 'solo0-30'
vmin, vmax = np.percentile(obs_all[ALVO_FIG], [2, 98])
def grade(I, vals):
    ix = np.round((I.x - I.x.min()) / PIX).astype(int); iy = np.round((I.y.max() - I.y) / PIX).astype(int)
    out = np.full((iy.max() + 1, ix.max() + 1) + np.shape(vals)[1:], np.nan); out[iy, ix] = vals
    ok = np.zeros(out.shape[:2], bool); ok[iy, ix] = True
    buraco = ndi.binary_closing(ok, np.ones((3, 3))) & ~ok   # falhas de 1 pixel no mosaico
    _, (ri, ci) = ndi.distance_transform_edt(~ok, return_indices=True)
    out[buraco] = out[ri[buraco], ci[buraco]]; return out
fig_d = {}
for area, P in PAIS.items():
    I = P['I']
    rgb = I[['bR', 'bG', 'bB']].values; rgb = np.clip(rgb / np.nanpercentile(rgb, 98), 0, 1)
    rgb = grade(I, np.c_[rgb, np.ones(len(I))]); rgb[np.isnan(rgb)] = 0
    virt = grade(I, P['alvo'][ALVO_FIG]['y']['observado'])
    ext = [I.x.min() - PIX / 2, I.x.max() + PIX / 2, I.y.min() - PIX / 2, I.y.max() + PIX / 2]
    fig_d[area] = dict(rgb=rgb, virt=virt, ext=ext, asp=(ext[3] - ext[2]) / (ext[1] - ext[0]))
hr = [max(d['asp'], .35) for d in fig_d.values()]
fig, ax = plt.subplots(3, 2, figsize=(7.2, 7.2 * .5 * sum(hr) + 1.0), gridspec_kw=dict(height_ratios=hr), constrained_layout=True)
for i, (area, d) in enumerate(fig_d.items()):
    g, yv = PAIS[area]['pontos'], PAIS[area]['obs'][ALVO_FIG]
    ax[i, 0].imshow(d['rgb'], extent=d['ext'])
    ax[i, 0].scatter(g.geometry.x, g.geometry.y, c=yv, cmap=SOC, vmin=vmin, vmax=vmax, s=26, edgecolors='#ffffff', linewidths=.8, zorder=3)
    im = ax[i, 1].imshow(d['virt'], extent=d['ext'], cmap=SOC, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax[i, 1].scatter(g.geometry.x, g.geometry.y, s=8, facecolors='none', edgecolors=INK, linewidths=.5, zorder=3)
    for j, A in enumerate(ax[i]):
        A.xaxis.set_major_formatter(fmt); A.yaxis.set_major_formatter(fmt)
        A.xaxis.set_major_locator(MaxNLocator(4)); A.yaxis.set_major_locator(MaxNLocator(3))
        A.tick_params(labelsize=5.5, length=2)
        if j == 1: A.set_yticklabels([])
        tit = f'{NOME_EN[area]}: real landscape' if j == 0 else 'Virtual landscape (simulated SOC stock)'
        A.set_title(f"{'abcdef'[i * 2 + j]}   {tit}", loc='left')
    ax[i, 0].set_ylabel('Northing (m)', fontsize=6)
for A in ax[2]: A.set_xlabel('Easting (m)', fontsize=6)
cb = fig.colorbar(im, ax=ax, orientation='horizontal', shrink=.5, aspect=35, pad=.01)
cb.set_label(f'SOC stock {ROT[ALVO_FIG]} (Mg ha⁻¹)'); cb.outline.set_visible(False)
fig.suptitle('Real landscape (UAV true-colour mosaic and measured SOC at the sampling points) and virtual landscape\n'
             '(SOC simulated at 2.5 m with the observed distribution and variogram; open circles = field points). SIRGAS 2000 / UTM 23S', fontsize=6.5)
for e in ('png', 'pdf'): fig.savefig(os.path.join(SAIDA, f'B3b_real_vs_virtual_landscape.{e}'))
plt.show()

## B4. Amostragem virtual
Sorteia `n` pixels, treina Ridge e Random Forest e avalia contra o mapa verdadeiro. É a etapa demorada: o tempo cresce com `REPS` e com o número de camadas em `ALVOS`.

In [ ]:
linhas = []
for area, P in PAIS.items():
    X = P['I'][IDX].values
    for alvo, Q in P['alvo'].items():
        for cen, y in Q['y'].items():
            aval = rng.choice(len(X), min(N_AVAL, len(X)), replace=False)
            for n in NS:
                for rep in range(REPS):
                    idx = rng.choice(len(X), n, replace=False)
                    for nome, mdl in (('Ridge', ridge()),
                                      ('RF', RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                                                                   random_state=rep, n_jobs=-1))):
                        mdl.fit(X[idx], y[idx]); p = mdl.predict(X[aval])
                        linhas.append(dict(area=area, camada=alvo, cenario=cen, R2_teto=Q['teto'][cen], n=n,
                                           rep=rep, modelo=nome, R2=r2_score(y[aval], p),
                                           RMSE=mean_squared_error(y[aval], p) ** 0.5))
        print(area, ROT[alvo], 'ok')
df = pd.DataFrame(linhas); df.to_csv(os.path.join(SAIDA, 'B4_resultados_simulacao.csv'), index=False)

## B5. Curvas e n mínimo
`n mínimo` = menor n em que pelo menos 80% das repetições atingem 80% do R² máximo possível no cenário.

In [ ]:
g = df.groupby(['area', 'camada', 'cenario', 'modelo', 'n'])
resumo = g.agg(R2_mediana=('R2', 'median'), R2_p10=('R2', lambda v: np.quantile(v, .1)),
               R2_teto=('R2_teto', 'first'), RMSE_mediana=('RMSE', 'median')).reset_index()
resumo['P_sucesso'] = g.apply(lambda d: np.mean(d.R2 >= FRAC * d.R2_teto)).values
resumo.loc[resumo.R2_teto < 0.05, 'P_sucesso'] = 0          # sem sinal: nenhum n é suficiente
resumo.to_csv(os.path.join(SAIDA, 'B5_curvas_resumo.csv'), index=False)

nmin = resumo[resumo.P_sucesso >= PROB].groupby(['area', 'camada', 'cenario', 'modelo']).n.min()
todas = pd.MultiIndex.from_product([list(AREAS), ALVOS, [str(c) for c in CENARIOS], ['Ridge', 'RF']],
                                   names=['area', 'camada', 'cenario', 'modelo'])
nmin = nmin.reindex(todas).map(lambda v: f'>{max(NS)}' if pd.isna(v) else str(int(v))).rename('n_min').reset_index()

# tabela em inglês para o artigo
tab = nmin.assign(col=nmin.area.map(SIGLA_EN) + ' ' + nmin.modelo.replace({'RF': 'RF'}),
                  layer=nmin.camada.map(ROT),
                  signal=nmin.cenario.map(lambda c: 'Observed' if c == 'observado' else f'R² = {c}'))
tab = tab.pivot(index=['layer', 'signal'], columns='col', values='n_min')
tab = tab[[f'{SIGLA_EN[a]} {m}' for a in AREAS for m in ['Ridge', 'RF']]]
tab = tab.reindex([ROT[c] for c in ALVOS], level=0)
tab.index.names = ['Layer', 'Signal strength']
tab.to_csv(os.path.join(SAIDA, 'B5_table_minimum_sample_size.csv'))
tab

## B6. Curvas de aprendizado (em inglês)
Uma figura por camada.

In [ ]:
cen = [str(c) for c in CENARIOS]
cores = dict(zip(cen, ['#7f7f7f', '#9ecae1', '#3182bd', '#08519c']))
for alvo in ALVOS:
    fig, ax = plt.subplots(2, 3, figsize=(7.2, 4.4), sharex=True, sharey=True, constrained_layout=True)
    for j, area in enumerate(AREAS):
        for i, mod in enumerate(['Ridge', 'RF']):
            A = ax[i, j]
            for c in cen:
                d = resumo[(resumo.area == area) & (resumo.camada == alvo) & (resumo.modelo == mod) & (resumo.cenario == c)]
                r2o = PAIS[area]['alvo'][alvo]['r2_obs']
                lab = f'Observed (R² = {r2o:.2f})' if c == 'observado' else f'R² = {float(c):.1f}'
                A.plot(d.n, d.R2_mediana, '-o', ms=2.5, lw=1.2, color=cores[c], label=lab)
                A.fill_between(d.n, d.R2_p10, d.R2_mediana, color=cores[c], alpha=.12, lw=0)
            A.axhline(0, ls='--', lw=.6, color=INK)
            A.set_xscale('log'); A.set_xticks(NS); A.set_xticklabels(NS, fontsize=5.5); A.minorticks_off()
            A.set_ylim(-0.6, 0.8); A.grid(color='#e2e2df', lw=.5)
            A.set_title(f"{NOME_EN[area]} · {'Ridge regression' if mod == 'Ridge' else 'Random forest'}", loc='left', fontsize=6.5)
            if j == 0: A.set_ylabel('R² against the true map')
            if i == 1: A.set_xlabel('Sample size (n)')
        ax[0, 2].legend(title='Signal strength', fontsize=5.5, title_fontsize=6, frameon=False)
    fig.suptitle(f'Learning curves in the virtual landscape, SOC stock {ROT[alvo]} (median and 10th percentile of {REPS} repetitions)', fontsize=7.5)
    nome = alvo.replace('solo', 'SOC_')
    fig.savefig(os.path.join(SAIDA, f'B6_learning_curves_{nome}.png')); fig.savefig(os.path.join(SAIDA, f'B6_learning_curves_{nome}.pdf'))
    plt.show()

# Parte C – relevo (MDE de 30 m)
Como os projetos do Metashape não existem mais, o relevo vem do Copernicus GLO-30 (modelo de superfície, gratuito). Se o arquivo `dem30_copernicus_utm23s.tif` já estiver na pasta `relevo`, ele é usado direto; senão, a célula baixa o recorte da AWS (precisa de internet) e reprojeta para SIRGAS 2000 / UTM 23S.
## C1. MDE

In [ ]:
from rasterio.warp import reproject
from rasterio.transform import from_origin
from scipy.ndimage import uniform_filter, map_coordinates
RELEVO = os.path.join(SAIDA, 'terrain'); os.makedirs(RELEVO, exist_ok=True)
DEM = os.path.join(DADOS, 'dem', 'dem30_copernicus_utm23s.tif')
PTS = {a: gpd.read_file(shp).to_crs(31983) for a, (_, shp) in AREAS.items()}
if not os.path.exists(DEM):
    import urllib.request
    T = 'Copernicus_DSM_COG_10_S04_00_W043_00_DEM'
    tmp = os.path.join(DADOS, 'dem', 'cop_S04W043.tif')
    urllib.request.urlretrieve(f'https://copernicus-dem-30m.s3.amazonaws.com/{T}/{T}.tif', tmp)
    x0, y0, x1, y1 = pd.concat(PTS.values()).total_bounds + np.array([-3000, -3000, 3000, 3000]); RES = 30.0
    nx, ny = int((x1 - x0) / RES), int((y1 - y0) / RES); tr = from_origin(x0, y1, RES, RES)
    z = np.full((ny, nx), np.nan, 'float32')
    with rasterio.open(tmp) as s:
        reproject(rasterio.band(s, 1), z, dst_transform=tr, dst_crs='EPSG:31983', resampling=Resampling.bilinear)
    with rasterio.open(DEM, 'w', driver='GTiff', height=ny, width=nx, count=1, dtype='float32',
                       crs='EPSG:31983', transform=tr, compress='lzw') as d:
        d.write(z, 1)
    os.remove(tmp)
with rasterio.open(DEM) as s:
    Z = s.read(1).astype(float); TR = s.transform; RES = s.res[0]
print(Z.shape, f'pixel {RES:.0f} m, altitude {np.nanmin(Z):.0f} a {np.nanmax(Z):.0f} m')

## C2. Atributos do relevo nos pontos
Declividade, orientação (northness, eastness), curvatura em planta (Zevenbergen e Thorne), TPI (janela de ~330 m) e índice topográfico de umidade (TWI, fluxo D8).

In [ ]:
def d8_acumulacao(z):
    ny, nx = z.shape; acc = np.ones(z.size); zf = z.ravel()
    viz = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    rr, cc = np.divmod(np.arange(z.size), nx); rec = np.full(z.size, -1); best = np.zeros(z.size)
    for dr, dc in viz:
        r2, c2 = rr + dr, cc + dc; ok = (r2 >= 0) & (r2 < ny) & (c2 >= 0) & (c2 < nx)
        j = np.where(ok, r2 * nx + c2, 0); queda = np.where(ok, (zf - zf[j]) / np.hypot(dr, dc), 0)
        m = queda > best; best[m] = queda[m]; rec[m] = j[m]
    for i in np.argsort(-zf):
        if rec[i] >= 0: acc[rec[i]] += acc[i]
    return acc.reshape(z.shape)
zp = np.pad(Z, 1, mode='edge'); ny, nx = Z.shape
S = lambda i, j: zp[1 + i:1 + i + ny, 1 + j:1 + j + nx]
p = (S(0, 1) - S(0, -1)) / (2 * RES); q = (S(-1, 0) - S(1, 0)) / (2 * RES)
r = (S(0, 1) - 2 * Z + S(0, -1)) / RES**2; t = (S(-1, 0) - 2 * Z + S(1, 0)) / RES**2
s_ = (S(-1, 1) - S(-1, -1) - S(1, 1) + S(1, -1)) / (4 * RES**2)
decl = np.degrees(np.arctan(np.hypot(p, q))); asp = np.arctan2(-p, -q); pq = p**2 + q**2 + 1e-12
ATR = {'elevation': Z, 'slope': decl, 'northness': np.cos(asp), 'eastness': np.sin(asp),
       'plan_curv': -(q**2 * r - 2 * p * q * s_ + p**2 * t) / pq**1.5,
       'TPI': Z - uniform_filter(Z, size=11),
       'TWI': np.log(d8_acumulacao(Z) * RES / np.tan(np.radians(np.maximum(decl, 0.1))))}
REL = list(ATR)
lin = []
for a, g in PTS.items():
    cc, rr = ~TR * (g.geometry.x.values, g.geometry.y.values)
    d = {k: map_coordinates(v, [rr - .5, cc - .5], order=1) for k, v in ATR.items()}
    obs = obs_all[obs_all.tratamento == a].reset_index(drop=True)
    lin.append(pd.concat([obs, pd.DataFrame(d)], axis=1))
DR = pd.concat(lin, ignore_index=True); DR.to_csv(os.path.join(RELEVO, 'C2_atributos_relevo_pontos.csv'), index=False)
DR.groupby('tratamento')[REL].agg(['min', 'max']).round(2).T

## C3. O relevo explica o ECOS?
- Correlação de Spearman entre cada atributo e o ECOS, por área e camada, com correção de Benjamini-Hochberg (FDR).
- R² em validação cruzada (5 grupos, 20 repetições, Ridge) com índices espectrais, relevo e os dois juntos.

In [ ]:
from sklearn.model_selection import RepeatedKFold
REL_CV = [v for v in REL if v != 'profile_curv']   # curvatura de perfil quase constante no modelo de 30 m
def bh(p):
    p = np.asarray(p); o = np.argsort(p); q = p[o] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]; out = np.empty_like(q); out[o] = np.minimum(q, 1); return out
cor = []
for a, g in DR.groupby('tratamento'):
    for c in CAMADAS:
        for v in REL_CV:
            rs, pv = stats.spearmanr(g[v], g[c]); cor.append(dict(area=a, camada=ROT[c], atributo=v, rs=rs, p=pv))
cor = pd.DataFrame(cor); cor['p_BH'] = bh(cor.p)
cor.to_csv(os.path.join(RELEVO, 'C3_correlacoes_relevo_ECOS.csv'), index=False)
print('Menores p (com e sem correção):'); display(cor.sort_values('p').head(8).round(3))

def r2_cv(X, y, reps=20):
    out = []
    for rep in range(reps):
        m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 30)))
        pr = cross_val_predict(m, X, y, cv=RepeatedKFold(n_splits=5, n_repeats=1, random_state=rep))
        out.append(r2_score(y, pr))
    return np.median(out)
res = []
for a, g in DR.groupby('tratamento'):
    for c in CAMADAS:
        y = g[c].values
        res.append(dict(area=a, camada=ROT[c], R2_espectral=r2_cv(g[IDX].values, y), R2_relevo=r2_cv(g[REL_CV].values, y),
                        R2_espectral_relevo=r2_cv(g[IDX + REL_CV].values, y),
                        R2_elev_TWI=r2_cv(g[['elevation', 'TWI']].values, y)))
res = pd.DataFrame(res).round(2); res.to_csv(os.path.join(RELEVO, 'C3_r2_cv_relevo.csv'), index=False)
res